In [4]:
# %% [markdown]
# # Underwater Image Enhancement - Model Test
# Testing U-Net model architecture and forward pass

# %% [markdown]
# ## 1. Setup and Imports

# %%
import tensorflow as tf
import numpy as np
import os
import sys
from pathlib import Path

# Add current directory to path
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Current directory: {current_dir}")

# Check if modules exist and create if needed
def ensure_module(module_name, file_content):
    """Ensure a module file exists"""
    filepath = f"{module_name.replace('.', '/')}.py"
    dirpath = os.path.dirname(filepath)
    
    if dirpath and not os.path.exists(dirpath):
        os.makedirs(dirpath, exist_ok=True)
        with open(os.path.join(dirpath, '__init__.py'), 'w') as f:
            f.write('# Package init\n')
    
    if not os.path.exists(filepath):
        with open(filepath, 'w') as f:
            f.write(file_content)
        print(f"✅ Created {filepath}")
        return True
    return False

# Create models/basic_unet.py if it doesn't exist
basic_unet_content = '''"""
Basic U-Net model for image enhancement
"""
import tensorflow as tf
from tensorflow.keras import layers, Model

def conv_block(input_tensor, num_filters, kernel_size=3):
    """Convolutional block: Conv -> BatchNorm -> ReLU"""
    x = layers.Conv2D(num_filters, kernel_size, padding='same')(input_tensor)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x

def encoder_block(input_tensor, num_filters):
    """Encoder block: Two conv layers + MaxPool"""
    x = conv_block(input_tensor, num_filters)
    x = conv_block(x, num_filters)
    skip = x
    x = layers.MaxPool2D(pool_size=(2, 2))(x)
    return x, skip

def decoder_block(input_tensor, skip_tensor, num_filters):
    """Decoder block: UpConv -> Concatenate -> Two conv layers"""
    x = layers.Conv2DTranspose(num_filters, kernel_size=2, strides=2, padding='same')(input_tensor)
    x = layers.Concatenate()([x, skip_tensor])
    x = conv_block(x, num_filters)
    x = conv_block(x, num_filters)
    return x

def build_basic_unet(input_shape=(256, 256, 3)):
    """Build basic U-Net model"""
    inputs = layers.Input(shape=input_shape)
    
    # Encoder
    x1, skip1 = encoder_block(inputs, 32)
    x2, skip2 = encoder_block(x1, 64)
    x3, skip3 = encoder_block(x2, 128)
    x4, skip4 = encoder_block(x3, 256)
    
    # Bottleneck
    bottleneck = conv_block(x4, 512)
    bottleneck = conv_block(bottleneck, 512)
    
    # Decoder
    d4 = decoder_block(bottleneck, skip4, 256)
    d3 = decoder_block(d4, skip3, 128)
    d2 = decoder_block(d3, skip2, 64)
    d1 = decoder_block(d2, skip1, 32)
    
    # Output
    outputs = layers.Conv2D(3, 1, activation='sigmoid')(d1)
    
    model = Model(inputs, outputs, name='Basic_U-Net')
    return model
'''

# Create losses/simple_losses.py if it doesn't exist
simple_losses_content = '''"""
Simple loss functions for image enhancement
"""
import tensorflow as tf

class SimpleLosses:
    """Collection of basic loss functions"""
    
    @staticmethod
    def mse_loss(y_true, y_pred):
        """Mean Squared Error Loss"""
        return tf.reduce_mean(tf.square(y_true - y_pred))
    
    @staticmethod
    def mae_loss(y_true, y_pred):
        """Mean Absolute Error Loss"""
        return tf.reduce_mean(tf.abs(y_true - y_pred))
    
    @staticmethod
    def ssim_loss(y_true, y_pred, max_val=1.0):
        """Structural Similarity Loss"""
        return 1 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=max_val))
    
    @staticmethod
    def combined_loss(y_true, y_pred, alpha=0.5):
        """Combine MSE and SSIM losses"""
        mse = SimpleLosses.mse_loss(y_true, y_pred)
        ssim = SimpleLosses.ssim_loss(y_true, y_pred)
        return mse + alpha * ssim
'''

# Ensure modules exist
ensure_module('models.basic_unet', basic_unet_content)
ensure_module('losses.simple_losses', simple_losses_content)

# Now import
try:
    from models.basic_unet import build_basic_unet
    from losses.simple_losses import SimpleLosses
    print("\n✅ All modules imported successfully!")
except Exception as e:
    print(f"\n❌ Import error: {e}")
    raise

# %% [markdown]
# ## 2. Build Model

# %%
# Build model
model = build_basic_unet(input_shape=(256, 256, 3))

# Compile with Adam optimizer and combined loss
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=SimpleLosses.combined_loss,
    metrics=['mae', 'mse']
)

print("✅ Model compiled successfully!")
print(f"Total parameters: {model.count_params():,}")

# %% [markdown]
# ## 3. Model Architecture

# %%
# Display model summary
model.summary()

# Save model summary
os.makedirs('results', exist_ok=True)
with open('results/model_summary.txt', 'w') as f:
    model.summary(print_fn=lambda x: f.write(x + '\n'))
print("\n✅ Model summary saved to results/model_summary.txt")

# %% [markdown]
# ## 4. Test Forward Pass

# %%
# Test prediction
test_input = tf.random.normal([1, 256, 256, 3])
prediction = model.predict(test_input, verbose=0)

print(f"\n📊 Test Prediction Results:")
print(f"   Input shape: {test_input.shape}")
print(f"   Output shape: {prediction.shape}")
print(f"   Output range: [{prediction.min():.4f}, {prediction.max():.4f}]")

# %% [markdown]
# ## 5. Test Loss Functions

# %%
# Create sample data
y_true = tf.random.uniform([4, 256, 256, 3])
y_pred = tf.random.uniform([4, 256, 256, 3])

# Test loss functions
print("\n📉 Loss Function Tests:")
mse = SimpleLosses.mse_loss(y_true, y_pred)
mae = SimpleLosses.mae_loss(y_true, y_pred)
ssim = SimpleLosses.ssim_loss(y_true, y_pred)
combined = SimpleLosses.combined_loss(y_true, y_pred)

print(f"   MSE Loss:      {mse:.4f}")
print(f"   MAE Loss:      {mae:.4f}")
print(f"   SSIM Loss:     {ssim:.4f}")
print(f"   Combined Loss: {combined:.4f}")

# %% [markdown]
# ## 6. Summary

# %%
print("\n" + "="*60)
print("TEST SUMMARY")
print("="*60)
print(f"✅ Model built successfully")
print(f"✅ Model compiled with combined loss")
print(f"✅ Forward pass working")
print(f"✅ Loss functions tested")
print("="*60)

TensorFlow version: 2.20.0
NumPy version: 2.2.6
Current directory: /home/soham/Desktop/Projectsss/Underwater-Project
✅ Created models/basic_unet.py
✅ Created losses/simple_losses.py

✅ All modules imported successfully!


2026-03-09 01:14:24.411065: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


✅ Model compiled successfully!
Total parameters: 7,771,939


Model: "Basic_U-Net"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 256,  │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256, 256,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 256, 256,  │      9,248 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        128 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 256, 256,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 128,  │          0 │ activation_1[0][… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 128, 128,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 128, 128,  │     36,928 │ activation_2[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 64,    │          0 │ activation_3[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 64, 64,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        512 │ conv2d_4[0][0]  

 Total params: 7,771,939 (29.65 MB)

 Trainable params: 7,766,051 (29.63 MB)

 Non-trainable params: 5,888 (23.00 KB)


✅ Model summary saved to results/model_summary.txt

📊 Test Prediction Results:
   Input shape: (1, 256, 256, 3)
   Output shape: (1, 256, 256, 3)
   Output range: [0.4136, 0.6572]

📉 Loss Function Tests:
   MSE Loss:      0.1666
   MAE Loss:      0.3333
   SSIM Loss:     0.9943
   Combined Loss: 0.6637

TEST SUMMARY
✅ Model built successfully
✅ Model compiled with combined loss
✅ Forward pass working
✅ Loss functions tested
